```
Hybrid MOdel Arcituture
                    PRODUCT DATASET
                          │
             ┌────────────┴────────────┐
             │                         │
             ▼                         ▼
      STRUCTURED DATA              TEXT BRANCH
             │                         │
      ┌──────┴──────┐          Product Name
      │             │          + Company
   Numeric      Categorical    + Country
      │             │                │
 Weight, Year   Industry,           ↓
                Protocol,       Sentence-BERT
                Stage-level          │
                   │              384-D
                   │                 │
                   │                PCA
                   │                 │
                   └───────┬─────────┘
                           ▼
                    FEATURE FUSION
                           │
                    ┌──────┴──────┐
                    ▼             ▼
                Random Forest   XGBoost
                    │             │
                    └──────┬──────┘
                           ▼
                      Predicted PCF

  ```

In [7]:
# ============================================================
# 1. INSTALL REQUIRED PACKAGES
# ============================================================

!pip install -q sentence-transformers xgboost lightgbm catboost

In [8]:
# ============================================================
# 2. IMPORT LIBRARIES
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from scipy.sparse import hstack, csr_matrix

from sentence_transformers import SentenceTransformer

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
    FunctionTransformer
)
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

print("All libraries imported successfully.")

All libraries imported successfully.


In [9]:
# ============================================================
# 3. LOAD CARBON CATALOGUE PRODUCT DATASET
# ============================================================

DATA_PATH = "/content/PublicTablesForCarbonCatalogueDataDescriptor_v30Oct2021(Product Level Data)(4).csv"

df = pd.read_csv(
    DATA_PATH,
    encoding="latin1"
)

print("Dataset shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

display(df.head())

Dataset shape: (866, 25)

Columns:
['*PCF-ID', 'Year of reporting', '*Stage-level CO2e available', 'Product name (and functional unit)', 'Product detail', 'Company', 'Country (where company is incorporated)', "Company's GICS Industry Group", "Company's GICS Industry", "*Company's sector", 'Product weight (kg)', '*Source for product weight', "Product's carbon footprint (PCF, kg CO2e)", '*Carbon intensity', 'Protocol used for PCF', 'Relative change in PCF vs previous', 'Company-reported reason for change', '*Change reason category', '*%Upstream estimated from %Operations', '*Upstream CO2e (fraction of total PCF)', '*Operations CO2e (fraction of total PCF)', '*Downstream CO2e (fraction of total PCF)', '*Transport CO2e (fraction of total PCF)', '*EndOfLife CO2e (fraction of total PCF)', '*Adjustments to raw data (if any)']


,*PCF-ID,Year of reporting,*Stage-level CO2e available,Product name (and functional unit),Product detail,Company,Country (where company is incorporated),Company's GICS Industry Group,Company's GICS Industry,*Company's sector,...,Relative change in PCF vs previous,Company-reported reason for change,*Change reason category,*%Upstream estimated from %Operations,*Upstream CO2e (fraction of total PCF),*Operations CO2e (fraction of total PCF),*Downstream CO2e (fraction of total PCF),*Transport CO2e (fraction of total PCF),*EndOfLife CO2e (fraction of total PCF),*Adjustments to raw data (if any)
0,10056-1-2014,2014,Yes,Frosted Flakes(R) Cereal,"Frosted Flakes(R), 23 oz., Produced in Lancast...",Kellogg Company,USA,"Food, Beverage & Tobacco",Food Products,Food & Beverage,...,(not reported by company),N/a,N/a (no %change reported),No,57.50%,30.00%,12.50%,4.50%,(included in downstream but not reported separ...,Divided stage and total emissions by 1000 (bas...
1,10056-1-2015,2015,Yes,"Frosted Flakes, 23 oz, produced in Lancaster, ...",Cereal,Kellogg Company,USA,Food & Beverage Processing,Not used for 2015 reporting,Food & Beverage,...,(not reported by company),N/a,N/a (no %change reported),No,57.50%,30.00%,12.50%,4.50%,(included in downstream but not reported separ...,Divided stage and total emissions by 1000 (bas...
2,10222-1-2013,2013,Yes,Office Chair,Field not included in 2013 data,KNOLL INC,USA,Capital Goods,Building Products,Comm. equipm. & capital goods,...,(not reported by company),N/a,N/a (no previous data available),Yes,80.63%,17.36%,2.01%,(included in up/downstream but not reported se...,0.00%,"Changed %change to zero, according to field ""c..."
3,10261-1-2017,2017,Yes,Multifunction Printers,bizhub C458,"Konica Minolta, Inc.",Japan,Technology Hardware & Equipment,"Electronic Equipment, Instruments & Components","Computer, IT & telecom",...,(not reported by company),N/a,N/a (no previous data available),No,30.65%,5.51%,63.84%,1.01%,2.76%,NaN
4,10261-2-2017,2017,Yes,Multifunction Printers,bizhub C558,"Konica Minolta, Inc.",Japan,Technology Hardware & Equipment,"Electronic Equipment, Instruments & Components","Computer, IT & telecom",...,(not reported by company),N/a,N/a (no previous data available),No,25.08%,4.51%,70.41%,0.83%,2.26%,NaN


In [10]:
# ============================================================
# 4. DEFINE FINAL HYBRID ARCHITECTURE
# ============================================================

TARGET = "Product's carbon footprint (PCF, kg CO2e)"

# ------------------------------------------------------------
# TEXT BRANCH
# ------------------------------------------------------------

TEXT_FEATURES = [
    "Product name (and functional unit)",
    "Company",
    "Country (where company is incorporated)"
]

# ------------------------------------------------------------
# STRUCTURED NUMERICAL FEATURES
# ------------------------------------------------------------

NUMERIC_FEATURES = [
    "Year of reporting",
    "Product weight (kg)"
]

# ------------------------------------------------------------
# STRUCTURED CATEGORICAL FEATURES
# ------------------------------------------------------------

CATEGORICAL_FEATURES = [
    "Company's GICS Industry",
    "Protocol used for PCF",
    "*Stage-level CO2e available"
]

STRUCTURED_FEATURES = (
    NUMERIC_FEATURES +
    CATEGORICAL_FEATURES
)

# ------------------------------------------------------------
# Required columns
# ------------------------------------------------------------

REQUIRED_COLUMNS = (
    [TARGET]
    + TEXT_FEATURES
    + STRUCTURED_FEATURES
)

missing_columns = [
    col for col in REQUIRED_COLUMNS
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns:\n{missing_columns}"
    )

print("TEXT BRANCH:")
for col in TEXT_FEATURES:
    print("  -", col)

print("\nSTRUCTURED NUMERICAL:")
for col in NUMERIC_FEATURES:
    print("  -", col)

print("\nSTRUCTURED CATEGORICAL:")
for col in CATEGORICAL_FEATURES:
    print("  -", col)

print("\nTARGET:")
print("  -", TARGET)

TEXT BRANCH:
  - Product name (and functional unit)
  - Company
  - Country (where company is incorporated)

STRUCTURED NUMERICAL:
  - Year of reporting
  - Product weight (kg)

STRUCTURED CATEGORICAL:
  - Company's GICS Industry
  - Protocol used for PCF
  - *Stage-level CO2e available

TARGET:
  - Product's carbon footprint (PCF, kg CO2e)


In [11]:
# ============================================================
# 5. PREPARE DATASET
# ============================================================

data = df[REQUIRED_COLUMNS].copy()

# ------------------------------------------------------------
# Numeric conversion
# ------------------------------------------------------------

data[TARGET] = pd.to_numeric(
    data[TARGET],
    errors="coerce"
)

data["Product weight (kg)"] = pd.to_numeric(
    data["Product weight (kg)"],
    errors="coerce"
)

data["Year of reporting"] = pd.to_numeric(
    data["Year of reporting"],
    errors="coerce"
)

# ------------------------------------------------------------
# Clean text fields
# ------------------------------------------------------------

for col in TEXT_FEATURES:

    data[col] = (
        data[col]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

# ------------------------------------------------------------
# Remove rows with missing target
# ------------------------------------------------------------

before = len(data)

data = data.dropna(
    subset=[TARGET]
).reset_index(drop=True)

after = len(data)

print("Rows before target filtering:", before)
print("Rows after target filtering :", after)
print("Rows removed                :", before - after)

Rows before target filtering: 866
Rows after target filtering : 866
Rows removed                : 0


In [12]:
# ============================================================
# 6. CREATE COMBINED TEXT FOR SENTENCE-BERT
# ============================================================

data["Combined_Text"] = (
    "Product: "
    + data["Product name (and functional unit)"]
    + " | Company: "
    + data["Company"]
    + " | Country: "
    + data["Country (where company is incorporated)"]
)

display(
    data[
        TEXT_FEATURES + ["Combined_Text"]
    ].head(10)
)

,Product name (and functional unit),Company,Country (where company is incorporated),Combined_Text
0,Frosted Flakes(R) Cereal,Kellogg Company,USA,Product: Frosted Flakes(R) Cereal | Company: K...
1,"Frosted Flakes, 23 oz, produced in Lancaster, ...",Kellogg Company,USA,"Product: Frosted Flakes, 23 oz, produced in La..."
2,Office Chair,KNOLL INC,USA,Product: Office Chair | Company: KNOLL INC | C...
3,Multifunction Printers,"Konica Minolta, Inc.",Japan,Product: Multifunction Printers | Company: Kon...
4,Multifunction Printers,"Konica Minolta, Inc.",Japan,Product: Multifunction Printers | Company: Kon...
5,Multifunction Printers,"Konica Minolta, Inc.",Japan,Product: Multifunction Printers | Company: Kon...
6,KURALON fiber,"Kuraray Co., Ltd.",Japan,"Product: KURALON fiber | Company: Kuraray Co.,..."
7,Portland Cement,Lafarge S.A.,France,Product: Portland Cement | Company: Lafarge S....
8,Regular Straight 505® Jeans  Steel (Water<Less),Levi Strauss & Co.,USA,Product: Regular Straight 505® Jeans  Steel (...
9,Regular Straight 505® Jeans  Steel (Water<Less),Levi Strauss & Co.,USA,Product: Regular Straight 505® Jeans  Steel (...


In [41]:
# ============================================================
# 7. PRODUCT WEIGHT SKEWNESS CHECK
# ============================================================

weight_train_check = pd.to_numeric(
    data["Product weight (kg)"],
    errors="coerce"
)

weight_skewness = weight_train_check.skew()

print(
    "Product Weight skewness:",
    round(weight_skewness, 4)
)

print(
    "Product Weight minimum:",
    weight_train_check.min()
)

print(
    "Product Weight maximum:",
    weight_train_check.max()
)

Product Weight skewness: 15.2826
Product Weight minimum: 0.00127018
Product Weight maximum: 600000.0


In [14]:
# ============================================================
# 8. LOG TRANSFORMATION DECISION
# ============================================================

WEIGHT_SKEW_THRESHOLD = 1.0

APPLY_LOG_WEIGHT = (
    weight_skewness > WEIGHT_SKEW_THRESHOLD
)

print(
    "Apply log1p(Product Weight):",
    APPLY_LOG_WEIGHT
)

if APPLY_LOG_WEIGHT:
    print(
        f"Reason: skewness = {weight_skewness:.4f} "
        f"> threshold = {WEIGHT_SKEW_THRESHOLD}"
    )
else:
    print(
        f"Reason: skewness = {weight_skewness:.4f} "
        f"<= threshold = {WEIGHT_SKEW_THRESHOLD}"
    )

Apply log1p(Product Weight): True
Reason: skewness = 15.2826 > threshold = 1.0


In [15]:
# ============================================================
# 9. FIXED 80/20 TRAIN / TEST SPLIT
# ============================================================

train_df, test_df = train_test_split(
    data,
    test_size=0.20,
    random_state=42
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Training rows:", len(train_df))
print("Testing rows :", len(test_df))

Training rows: 692
Testing rows : 174


In [42]:
# ============================================================
# 10. TARGET AND INPUT DATA
# ============================================================

y_train = train_df[TARGET].to_numpy()
y_test = test_df[TARGET].to_numpy()

X_train_structured = train_df[
    STRUCTURED_FEATURES
].copy()

X_test_structured = test_df[
    STRUCTURED_FEATURES
].copy()

text_train = train_df[
    "Combined_Text"
].tolist()

text_test = test_df[
    "Combined_Text"
].tolist()

print("y_train:", y_train.shape)
print("y_test :", y_test.shape)
print("X_train:", X_train_structured.shape)
print("X_test :", X_test_structured.shape)
print(
    "Training text records:",
    len(text_train)
)

print(
    "Testing text records:",
    len(text_test)
)

y_train: (692,)
y_test : (174,)
X_train: (692, 5)
X_test : (174, 5)
Training text records: 692
Testing text records: 174


In [17]:
# ============================================================
# 11. LOAD SENTENCE-BERT
# ============================================================

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print(
    "Sentence-BERT model loaded successfully."
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Sentence-BERT model loaded successfully.


In [18]:
# ============================================================
# 12. GENERATE SENTENCE-BERT EMBEDDINGS
# ============================================================

train_embeddings = embedding_model.encode(
    text_train,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

test_embeddings = embedding_model.encode(
    text_test,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(
    "\nTrain embedding shape:",
    train_embeddings.shape
)

print(
    "Test embedding shape :",
    test_embeddings.shape
)

Batches:   0%|          | 0/22 [00:00<?, ?it/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]


Train embedding shape: (692, 384)
Test embedding shape : (174, 384)


In [19]:
# ============================================================
# 13. INNER VALIDATION SPLIT
# ============================================================

indices = np.arange(
    len(train_df)
)

inner_train_idx, validation_idx = train_test_split(
    indices,
    test_size=0.20,
    random_state=42
)

print(
    "Inner training rows:",
    len(inner_train_idx)
)

print(
    "Validation rows:",
    len(validation_idx)
)

print(
    "Final test rows:",
    len(test_df)
)

Inner training rows: 553
Validation rows: 139
Final test rows: 174


In [20]:
# ============================================================
# 14. INNER TRAIN / VALIDATION DATA
# ============================================================

X_inner_train_structured = (
    X_train_structured
    .iloc[inner_train_idx]
    .copy()
)

X_validation_structured = (
    X_train_structured
    .iloc[validation_idx]
    .copy()
)

y_inner_train = (
    y_train[inner_train_idx]
)

y_validation = (
    y_train[validation_idx]
)

emb_inner_train = (
    train_embeddings[inner_train_idx]
)

emb_validation = (
    train_embeddings[validation_idx]
)

print(
    "Inner structured:",
    X_inner_train_structured.shape
)

print(
    "Validation structured:",
    X_validation_structured.shape
)

print(
    "Inner embeddings:",
    emb_inner_train.shape
)

print(
    "Validation embeddings:",
    emb_validation.shape
)

Inner structured: (553, 5)
Validation structured: (139, 5)
Inner embeddings: (553, 384)
Validation embeddings: (139, 384)


In [43]:
# ============================================================
# 15. LOG TRANSFORMATION FUNCTION
# ============================================================

def log1p_transform(X):

    X = np.asarray(
        X,
        dtype=float
    )

    if np.any(X < 0):
        raise ValueError(
            "Negative Product Weight detected. "
            "log1p cannot be applied."
        )

    return np.log1p(X)

In [22]:
# ============================================================
# 16. STRUCTURED PREPROCESSOR — TREE MODELS
# ============================================================

if APPLY_LOG_WEIGHT:

    numeric_tree_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "log_weight",
            FunctionTransformer(
                log1p_transform,
                validate=False
            )
        )
    ])

else:

    numeric_tree_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        )
    ])


categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="most_frequent"
        )
    ),
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True
        )
    )
])


tree_structured_preprocessor = ColumnTransformer([
    (
        "numeric",
        numeric_tree_pipeline,
        NUMERIC_FEATURES
    ),
    (
        "categorical",
        categorical_pipeline,
        CATEGORICAL_FEATURES
    )
])

In [23]:
# ============================================================
# 17. STRUCTURED PREPROCESSOR — RIDGE
# ============================================================

if APPLY_LOG_WEIGHT:

    numeric_ridge_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "log_weight",
            FunctionTransformer(
                log1p_transform,
                validate=False
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ])

else:

    numeric_ridge_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ])


ridge_structured_preprocessor = ColumnTransformer([
    (
        "numeric",
        numeric_ridge_pipeline,
        NUMERIC_FEATURES
    ),
    (
        "categorical",
        categorical_pipeline,
        CATEGORICAL_FEATURES
    )
])

In [24]:
# ============================================================
# 18. PCA CANDIDATES
# ============================================================

PCA_CANDIDATES = [
    50,
    75,
    100,
    150,
    200
]

print(
    "PCA candidates:",
    PCA_CANDIDATES
)

PCA candidates: [50, 75, 100, 150, 200]


In [25]:
# ============================================================
# 19. EVALUATION FUNCTION
# ============================================================

def evaluate_predictions(
    y_true,
    predictions
):

    mae = mean_absolute_error(
        y_true,
        predictions
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            predictions
        )
    )

    r2 = r2_score(
        y_true,
        predictions
    )

    return mae, rmse, r2

In [26]:
# ============================================================
# 20. DEFINE REGRESSION MODELS
# ============================================================

def create_tree_models():

    return {

        "Random Forest": RandomForestRegressor(
            n_estimators=300,
            random_state=42,
            n_jobs=-1
        ),

        "XGBoost": XGBRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            random_state=42,
            n_jobs=-1
        ),

        "LightGBM": LGBMRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=4,
            num_leaves=31,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            n_jobs=-1,
            verbosity=-1
        ),

        "CatBoost": CatBoostRegressor(
            iterations=300,
            learning_rate=0.05,
            depth=6,
            loss_function="RMSE",
            random_seed=42,
            verbose=False
        )
    }


def create_ridge():

    return Ridge(
        alpha=10.0
    )

In [27]:
# ============================================================
# 21. PCA + MODEL SELECTION
# ============================================================

selection_results = []

for n_components in PCA_CANDIDATES:

    print("\n" + "=" * 80)
    print(
        f"Testing PCA = {n_components}"
    )
    print("=" * 80)

    # --------------------------------------------------------
    # PCA
    # --------------------------------------------------------

    pca = PCA(
        n_components=n_components,
        random_state=42
    )

    emb_train_pca = (
        pca.fit_transform(
            emb_inner_train
        )
    )

    emb_validation_pca = (
        pca.transform(
            emb_validation
        )
    )

    explained_variance = (
        pca.explained_variance_ratio_
        .sum()
    )

    print(
        "Explained variance:",
        round(
            explained_variance,
            6
        )
    )

    # --------------------------------------------------------
    # STRUCTURED PREPROCESSING — TREE
    # --------------------------------------------------------

    X_inner_tree = (
        tree_structured_preprocessor
        .fit_transform(
            X_inner_train_structured
        )
    )

    X_validation_tree = (
        tree_structured_preprocessor
        .transform(
            X_validation_structured
        )
    )

    # --------------------------------------------------------
    # STRUCTURED PREPROCESSING — RIDGE
    # --------------------------------------------------------

    X_inner_ridge = (
        ridge_structured_preprocessor
        .fit_transform(
            X_inner_train_structured
        )
    )

    X_validation_ridge = (
        ridge_structured_preprocessor
        .transform(
            X_validation_structured
        )
    )

    # --------------------------------------------------------
    # EMBEDDINGS → SPARSE
    # --------------------------------------------------------

    emb_train_sparse = csr_matrix(
        emb_train_pca
    )

    emb_validation_sparse = csr_matrix(
        emb_validation_pca
    )

    # --------------------------------------------------------
    # FEATURE FUSION — TREE
    # --------------------------------------------------------

    X_inner_tree_hybrid = hstack([
        X_inner_tree,
        emb_train_sparse
    ]).tocsr()

    X_validation_tree_hybrid = hstack([
        X_validation_tree,
        emb_validation_sparse
    ]).tocsr()

    # --------------------------------------------------------
    # FEATURE FUSION — RIDGE
    # --------------------------------------------------------

    X_inner_ridge_hybrid = hstack([
        X_inner_ridge,
        emb_train_sparse
    ]).tocsr()

    X_validation_ridge_hybrid = hstack([
        X_validation_ridge,
        emb_validation_sparse
    ]).tocsr()

    print(
        "Hybrid feature count:",
        X_inner_tree_hybrid.shape[1]
    )

    # ========================================================
    # TREE MODELS
    # ========================================================

    tree_models = create_tree_models()

    for model_name, model in tree_models.items():

        print(
            f"  Training {model_name}..."
        )

        model.fit(
            X_inner_tree_hybrid,
            y_inner_train
        )

        predictions = model.predict(
            X_validation_tree_hybrid
        )

        mae, rmse, r2 = (
            evaluate_predictions(
                y_validation,
                predictions
            )
        )

        selection_results.append({
            "Representation": "Hybrid",
            "Model": model_name,
            "PCA Components": n_components,
            "Explained Variance": explained_variance,
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2
        })

    # ========================================================
    # RIDGE
    # ========================================================

    print("  Training Ridge...")

    ridge = create_ridge()

    ridge.fit(
        X_inner_ridge_hybrid,
        y_inner_train
    )

    ridge_predictions = ridge.predict(
        X_validation_ridge_hybrid
    )

    ridge_mae, ridge_rmse, ridge_r2 = (
        evaluate_predictions(
            y_validation,
            ridge_predictions
        )
    )

    selection_results.append({
        "Representation": "Hybrid",
        "Model": "Ridge",
        "PCA Components": n_components,
        "Explained Variance": explained_variance,
        "MAE": ridge_mae,
        "RMSE": ridge_rmse,
        "R2": ridge_r2
    })


Testing PCA = 50
Explained variance: 0.75771
Hybrid feature count: 106
  Training Random Forest...
  Training XGBoost...
  Training LightGBM...
  Training CatBoost...
  Training Ridge...

Testing PCA = 75
Explained variance: 0.850939
Hybrid feature count: 131
  Training Random Forest...
  Training XGBoost...
  Training LightGBM...
  Training CatBoost...
  Training Ridge...

Testing PCA = 100
Explained variance: 0.906513
Hybrid feature count: 156
  Training Random Forest...
  Training XGBoost...
  Training LightGBM...
  Training CatBoost...
  Training Ridge...

Testing PCA = 150
Explained variance: 0.964667
Hybrid feature count: 206
  Training Random Forest...
  Training XGBoost...
  Training LightGBM...
  Training CatBoost...
  Training Ridge...

Testing PCA = 200
Explained variance: 0.988054
Hybrid feature count: 256
  Training Random Forest...
  Training XGBoost...
  Training LightGBM...
  Training CatBoost...
  Training Ridge...


In [28]:
# ============================================================
# 22. DISPLAY MODEL / PCA RESULTS
# ============================================================

selection_results_df = pd.DataFrame(
    selection_results
)

selection_results_df = (
    selection_results_df
    .sort_values(
        "R2",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    selection_results_df.round(6)
)

,Representation,Model,PCA Components,Explained Variance,MAE,RMSE,R2
0,Hybrid,CatBoost,75,0.850939,4238.323028,15865.486375,0.526425
1,Hybrid,CatBoost,50,0.757710,4237.062997,16152.651298,0.509127
2,Hybrid,CatBoost,150,0.964667,4422.756725,16323.083038,0.498713
3,Hybrid,XGBoost,100,0.906513,3423.015968,17042.905728,0.453527
4,Hybrid,CatBoost,100,0.906513,4755.585899,17063.171660,0.452226
5,Hybrid,XGBoost,150,0.964667,3337.738999,17217.581940,0.442267
6,Hybrid,XGBoost,200,0.988054,3841.236274,18254.729821,0.373051
7,Hybrid,XGBoost,75,0.850939,3585.536641,19084.137801,0.314785
8,Hybrid,XGBoost,50,0.757710,3557.327096,19086.104973,0.314644
9,Hybrid,CatBoost,200,0.988054,5611.651596,23534.635866,-0.042070


In [29]:
# ============================================================
# 23. PCA PERFORMANCE COMPARISON
# ============================================================

pca_summary = (
    selection_results_df[
        [
            "Model",
            "PCA Components",
            "Explained Variance",
            "R2",
            "RMSE",
            "MAE"
        ]
    ]
    .sort_values(
        ["Model", "PCA Components"]
    )
)

display(
    pca_summary.round(6)
)

,Model,PCA Components,Explained Variance,R2,RMSE,MAE
1,CatBoost,50,0.757710,0.509127,16152.651298,4237.062997
0,CatBoost,75,0.850939,0.526425,15865.486375,4238.323028
4,CatBoost,100,0.906513,0.452226,17063.171660,4755.585899
2,CatBoost,150,0.964667,0.498713,16323.083038,4422.756725
9,CatBoost,200,0.988054,-0.042070,23534.635866,5611.651596
10,LightGBM,50,0.757710,-0.785870,30809.458984,17393.306499
11,LightGBM,75,0.850939,-1.374209,35523.734733,22102.379281
12,LightGBM,100,0.906513,-1.419932,35864.163935,24424.607994
14,LightGBM,150,0.964667,-1.691718,37824.561098,24820.615844
13,LightGBM,200,0.988054,-1.571117,36967.498789,25145.720637


In [30]:
# ============================================================
# 24. SELECT BEST CONFIGURATION
# ============================================================

best_config = (
    selection_results_df
    .sort_values(
        "R2",
        ascending=False
    )
    .iloc[0]
)

BEST_MODEL = best_config["Model"]

BEST_PCA = int(
    best_config["PCA Components"]
)

BEST_VALIDATION_R2 = (
    best_config["R2"]
)

print("=" * 80)
print("BEST HYBRID CONFIGURATION")
print("=" * 80)

print(
    "Model:",
    BEST_MODEL
)

print(
    "PCA components:",
    BEST_PCA
)

print(
    "Explained variance:",
    round(
        best_config["Explained Variance"],
        6
    )
)

print(
    "Validation MAE:",
    round(
        best_config["MAE"],
        6
    )
)

print(
    "Validation RMSE:",
    round(
        best_config["RMSE"],
        6
    )
)

print(
    "Validation R²:",
    round(
        BEST_VALIDATION_R2,
        6
    )
)

BEST HYBRID CONFIGURATION
Model: CatBoost
PCA components: 75
Explained variance: 0.850939
Validation MAE: 4238.323028
Validation RMSE: 15865.486375
Validation R²: 0.526425


In [31]:
# ============================================================
# 25. SAVE MODEL SELECTION RESULTS
# ============================================================

selection_results_df.to_csv(
    "04_hybrid_model_selection.csv",
    index=False
)

print(
    "Saved: 04_hybrid_model_selection.csv"
)

Saved: 04_hybrid_model_selection.csv


In [32]:
# ============================================================
# 26. FREEZE FINAL CONFIGURATION
# ============================================================

FROZEN_CONFIGURATION = {
    "random_state": 42,
    "test_size": 0.20,
    "text_features": TEXT_FEATURES,
    "structured_numeric": NUMERIC_FEATURES,
    "structured_categorical": CATEGORICAL_FEATURES,
    "embedding_model": "all-MiniLM-L6-v2",
    "original_embedding_dimensions": 384,
    "pca_components": BEST_PCA,
    "selected_model": BEST_MODEL,
    "log_product_weight": APPLY_LOG_WEIGHT
}

print("=" * 80)
print("FROZEN HYBRID CONFIGURATION")
print("=" * 80)

for key, value in FROZEN_CONFIGURATION.items():
    print(
        f"{key}: {value}"
    )

FROZEN HYBRID CONFIGURATION
random_state: 42
test_size: 0.2
text_features: ['Product name (and functional unit)', 'Company', 'Country (where company is incorporated)']
structured_numeric: ['Year of reporting', 'Product weight (kg)']
structured_categorical: ["Company's GICS Industry", 'Protocol used for PCF', '*Stage-level CO2e available']
embedding_model: all-MiniLM-L6-v2
original_embedding_dimensions: 384
pca_components: 75
selected_model: CatBoost
log_product_weight: True


In [33]:
# ============================================================
# 27. FINAL PCA
#     FIT ONLY ON OUTER TRAINING DATA
# ============================================================

final_pca = PCA(
    n_components=BEST_PCA,
    random_state=42
)

final_train_embeddings_pca = (
    final_pca.fit_transform(
        train_embeddings
    )
)

final_test_embeddings_pca = (
    final_pca.transform(
        test_embeddings
    )
)

final_explained_variance = (
    final_pca
    .explained_variance_ratio_
    .sum()
)

print(
    "Final train embedding shape:",
    final_train_embeddings_pca.shape
)

print(
    "Final test embedding shape:",
    final_test_embeddings_pca.shape
)

print(
    "Final explained variance:",
    round(
        final_explained_variance,
        6
    )
)

Final train embedding shape: (692, 75)
Final test embedding shape: (174, 75)
Final explained variance: 0.842202


In [34]:
# ============================================================
# 28. FINAL STRUCTURED PREPROCESSING
# ============================================================

final_tree_structured = (
    tree_structured_preprocessor
    .fit_transform(
        X_train_structured
    )
)

final_test_tree_structured = (
    tree_structured_preprocessor
    .transform(
        X_test_structured
    )
)

final_ridge_structured = (
    ridge_structured_preprocessor
    .fit_transform(
        X_train_structured
    )
)

final_test_ridge_structured = (
    ridge_structured_preprocessor
    .transform(
        X_test_structured
    )
)

In [35]:
# ============================================================
# 29. FINAL FEATURE FUSION
# ============================================================

train_embedding_sparse = csr_matrix(
    final_train_embeddings_pca
)

test_embedding_sparse = csr_matrix(
    final_test_embeddings_pca
)

# ------------------------------------------------------------
# TREE MODELS
# ------------------------------------------------------------

X_train_tree_hybrid = hstack([
    final_tree_structured,
    train_embedding_sparse
]).tocsr()

X_test_tree_hybrid = hstack([
    final_test_tree_structured,
    test_embedding_sparse
]).tocsr()

# ------------------------------------------------------------
# RIDGE
# ------------------------------------------------------------

X_train_ridge_hybrid = hstack([
    final_ridge_structured,
    train_embedding_sparse
]).tocsr()

X_test_ridge_hybrid = hstack([
    final_test_ridge_structured,
    test_embedding_sparse
]).tocsr()

print(
    "Tree hybrid train:",
    X_train_tree_hybrid.shape
)

print(
    "Tree hybrid test:",
    X_test_tree_hybrid.shape
)

Tree hybrid train: (692, 138)
Tree hybrid test: (174, 138)


In [36]:
# ============================================================
# 30. CREATE FROZEN FINAL MODEL
# ============================================================

if BEST_MODEL == "Random Forest":

    final_model = RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    )

    X_final_train = X_train_tree_hybrid
    X_final_test = X_test_tree_hybrid


elif BEST_MODEL == "XGBoost":

    final_model = XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )

    X_final_train = X_train_tree_hybrid
    X_final_test = X_test_tree_hybrid


elif BEST_MODEL == "LightGBM":

    final_model = LGBMRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )

    X_final_train = X_train_tree_hybrid
    X_final_test = X_test_tree_hybrid


elif BEST_MODEL == "CatBoost":

    final_model = CatBoostRegressor(
        iterations=300,
        learning_rate=0.05,
        depth=6,
        loss_function="RMSE",
        random_seed=42,
        verbose=False
    )

    X_final_train = X_train_tree_hybrid
    X_final_test = X_test_tree_hybrid


elif BEST_MODEL == "Ridge":

    final_model = Ridge(
        alpha=10.0
    )

    X_final_train = X_train_ridge_hybrid
    X_final_test = X_test_ridge_hybrid


else:

    raise ValueError(
        f"Unknown model: {BEST_MODEL}"
    )

In [37]:
# ============================================================
# 31. TRAIN FINAL FROZEN MODEL
# ============================================================

final_model.fit(
    X_final_train,
    y_train
)

print(
    f"{BEST_MODEL} trained on "
    f"{len(y_train)} training observations."
)

CatBoost trained on 692 training observations.


In [38]:
# ============================================================
# 32. FINAL TEST PREDICTION
# ============================================================

final_predictions = (
    final_model.predict(
        X_final_test
    )
)

final_mae, final_rmse, final_r2 = (
    evaluate_predictions(
        y_test,
        final_predictions
    )
)

print("=" * 80)
print("FINAL HYBRID TEST PERFORMANCE")
print("=" * 80)

print(
    "Model:",
    BEST_MODEL
)

print(
    "PCA components:",
    BEST_PCA
)

print(
    "MAE:",
    round(final_mae, 6)
)

print(
    "RMSE:",
    round(final_rmse, 6)
)

print(
    "R²:",
    round(final_r2, 6)
)

FINAL HYBRID TEST PERFORMANCE
Model: CatBoost
PCA components: 75
MAE: 2988.406308
RMSE: 6480.213641
R²: 0.723239


In [39]:
# ============================================================
# 33. FINAL RESULT TABLE
# ============================================================

final_results = pd.DataFrame([
    {
        "Representation": "Hybrid",
        "Text Features": (
            "Product Name + Company + Country"
        ),
        "Structured Features": (
            "Year + Product Weight + "
            "Industry + Protocol + Stage-level CO2e"
        ),
        "Model": BEST_MODEL,
        "PCA Components": BEST_PCA,
        "Explained Variance": final_explained_variance,
        "MAE": final_mae,
        "RMSE": final_rmse,
        "R2": final_r2
    }
])

display(
    final_results.round(6)
)

,Representation,Text Features,Structured Features,Model,PCA Components,Explained Variance,MAE,RMSE,R2
0,Hybrid,Product Name + Company + Country,Year + Product Weight + Industry + Protocol + ...,CatBoost,75,0.842202,2988.406308,6480.213641,0.723239


In [40]:
# ============================================================
# 34. SAVE FINAL EXPERIMENT RESULTS
# ============================================================

final_results.to_csv(
    "04_hybrid_final_test_result.csv",
    index=False
)

pd.DataFrame(
    [
        {
            "Setting": key,
            "Value": value
        }
        for key, value in FROZEN_CONFIGURATION.items()
    ]
).to_csv(
    "04_hybrid_frozen_configuration.csv",
    index=False
)

print("Saved:")
print(" - 04_hybrid_model_selection.csv")
print(" - 04_hybrid_final_test_result.csv")
print(" - 04_hybrid_frozen_configuration.csv")

Saved:
 - 04_hybrid_model_selection.csv
 - 04_hybrid_final_test_result.csv
 - 04_hybrid_frozen_configuration.csv


```

                         CARBON CATALOGUE
                                │
                                ▼
                    Fixed 80/20 train/test split
                                │
                 ┌──────────────┴──────────────┐
                 │                             │
                 ▼                             ▼
        STRUCTURED BRANCH                  TEXT BRANCH
                 │                             │
        ┌────────┴────────┐          ┌─────────┴─────────┐
        │                 │          │         │         │
      Year             Weight    Product     Company   Country
        │                 │       Name
        │              log1p          │
        │                 │          └───────┬───────┘
        │                 │                  │
        │            Categorical       Combined Text
        │                 │                  │
        │                 │          Sentence-BERT
        │                 │                  │
        │                 │                384-D
        │                 │                  │
        │                 │                 PCA
        │                 │                  │
        └────────┬────────┘                  │
                 │                           │
                 └─────────────┬─────────────┘
                               ▼
                        FEATURE FUSION
                               │
             ┌───────┬────────┼────────┬────────┐
             ▼       ▼        ▼        ▼        ▼
           Ridge     RF     XGBoost  LightGBM CatBoost
             │       │        │        │        │
             └───────┴────────┴────────┴────────┘
                               │
                               ▼
                       VALIDATION RESULTS
                               │
                               ▼
                 SELECT BEST PCA + MODEL
                               │
                               ▼
                          FREEZE
                               │
                               ▼
                    FINAL 20% TEST SET
                               │
                               ▼
                    FINAL HYBRID PERFORMANCE


          